# <a href="https://girafe.ai/" target="_blank" rel="noopener noreferrer"><img src="https://raw.githubusercontent.com/girafe-ai/ml-course/7096a5df4cada5ee651be1e3215c2f7fb8a7e0bf/logo_margin.svg" alt="girafe-ai logo" width="150px" align="left"></a> YMLS Uzbekistan, ML Fundamentals, fall 2026 <a class="tocSkip">

# Week 3 seminar. Regularisation and model validation

Polynomials on 16 flats show underfitting and overfitting. Then ridge regression from scratch, ridge and lasso on a listing with twin and junk columns, and cross-validation to choose the regularisation strength. At the end, real medical data with 65 features and a split that leaks.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/girafe-ai/ml-course/blob/26f_ymls_uz/week0_03_regularisation_and_validation/week0_03_01_regularisation_and_validation.ipynb)

This week's [lecture slides](https://github.com/girafe-ai/ml-course/blob/26f_ymls_uz/week0_03_regularisation_and_validation/ml-basic_lect03_regularisation_validation.pdf) are in the same folder. The links to the interactive pages are in the course chat.

_YMLS Uzbekistan, ML Fundamentals, fall 2026_

Fill in the cells marked `# YOUR CODE HERE`. The cell after each exercise checks your answer and says what is wrong. Run the cells from top to bottom, since later sections use earlier variables. It takes about 80 minutes.

## 0. Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import make_column_transformer
from sklearn.datasets import load_diabetes
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Lasso, LassoCV, LinearRegression, Ridge, RidgeCV
from sklearn.model_selection import (
    GridSearchCV,
    GroupKFold,
    KFold,
    cross_val_score,
    train_test_split,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures, StandardScaler


SEED = 42  # one seed for every split and random generator


def rmse(y_true, y_pred):
    """Root mean squared error, in the units of the target."""
    return np.sqrt(np.mean((y_true - y_pred) ** 2))


def three_digits(value):
    """A number rounded to three significant digits, as text: 3089.4 becomes 3,090."""
    return f"{float(f'{value:.3g}'):,.0f}"


# Plot style: gray axes, colour only for data
INK, INK2, INK3, LINE = "#1D1D1F", "#6E6E73", "#86868B", "#D2D2D7"
ACCENT, ORANGE, GREEN, PURPLE = "#0071E3", "#FF9F0A", "#34C759", "#AF52DE"
plt.rcParams.update(
    {
        "font.family": "sans-serif",
        "font.sans-serif": ["Helvetica Neue", "Arial", "DejaVu Sans"],
        "font.size": 13,
        "mathtext.fontset": "cm",
        "axes.edgecolor": LINE,
        "axes.linewidth": 0.8,
        "axes.labelcolor": INK2,
        "axes.titlecolor": INK,
        "axes.titlesize": 14,
        "axes.titleweight": "bold",
        "axes.titlelocation": "left",
        "axes.titlepad": 10,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": False,
        "axes.prop_cycle": plt.cycler(color=[ACCENT, ORANGE, GREEN, PURPLE, INK3]),
        "xtick.color": INK3,
        "ytick.color": INK3,
        "xtick.labelsize": 11,
        "ytick.labelsize": 11,
        "xtick.major.size": 0,
        "ytick.major.size": 0,
        "xtick.minor.size": 0,
        "ytick.minor.size": 0,
        "legend.frameon": False,
        "legend.fontsize": 11,
        "lines.linewidth": 2.2,
        "lines.markersize": 6,
        "figure.facecolor": "white",
    }
)

## 1. When a better fit is a worse model

Sixteen flats, each with a distance to the city centre $x$ in km and a price per m² in US dollars. The data is synthetic: price per m² $= 1750 - 150 x + 6.2 x^2 + \varepsilon$ with noise $\varepsilon \sim \mathcal{N}(0, 70^2)$. We also draw 200 new flats from the same city. A real project never has them. Here they show how a model does on flats it has not seen.

In [ ]:
def curve_sample(seed, n, sd=70):
    """n flats: the distance to the centre in km (sorted) and the price per m² in US dollars."""
    rng = np.random.default_rng(seed)
    x = np.round(np.sort(rng.uniform(0.5, 14, n)), 1)
    y = np.round(1750 - 150 * x + 6.2 * x**2 + rng.normal(0, sd, n))
    return x, y


dist, price_m2 = curve_sample(68, 16)  # the 16 flats we train on
dist_new, price_m2_new = curve_sample(999, 200)  # 200 new flats from the same city
x_grid = np.linspace(0.5, 14, 400)  # a fine grid of distances, to draw curves

print("distance, km:   ", dist[:6], "...")
print("price per m², $:", price_m2[:6], "...")

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(
    x_grid,
    1750 - 150 * x_grid + 6.2 * x_grid**2,
    color=INK3,
    ls="--",
    lw=1.5,
    label="the true curve",
)
ax.scatter(
    dist, price_m2, s=55, color=ACCENT, edgecolor="white", linewidth=0.8, zorder=3, label="16 flats"
)
ax.set(xlabel="distance to the centre, km", ylabel="price per m², $")
ax.legend()
plt.show()

### 1.1 Polynomial features

A model that is linear in the weights can still draw a curve, because the features are ours to choose. Rescale the distance to $t = (x - 7.25) / 7.25$, which lies in $[-1, 1]$, and take its powers as features:

$$\hat y = b + w_1 t + w_2 t^2 + \dots + w_d t^d.$$

The degree $d$ sets how flexible the curve is.

**Exercise 1.1.** Write `poly_features(x, degree)`: a matrix with the columns $t, t^2, \dots, t^d$ and no column of ones.

In [ ]:
def poly_features(x, degree):
    """Columns t, t^2, ..., t^degree with t = (x - 7.25) / 7.25. No column of ones."""
    # YOUR CODE HERE: compute t from x, then stack the columns t**1, t**2, ..., t**degree
    # (np.column_stack takes a list of columns)
    return None

In [ ]:
features_3 = poly_features(dist, 3)
assert features_3 is not None, "Fill in poly_features above"
t_check = (dist - 7.25) / 7.25
assert features_3.shape == (16, 3), "Expect one row per flat and one column per power: (16, 3)"
assert np.allclose(features_3[:, 0], t_check), "The first column is t itself"
assert np.allclose(features_3[:, 2], t_check**3), "Column number k holds t to the power k"
print("the first three flats, columns t, t^2, t^3:")
print(features_3[:3].round(3))

Least squares on these features is last week's `np.linalg.lstsq` with a column of ones for the intercept. Fit three degrees.

In [ ]:
def fit_poly(x, y, degree):
    """Least squares polynomial: the intercept b and the weights of t, ..., t^degree."""
    features = poly_features(x, degree)
    assert features is not None, "Finish Exercise 1.1 first"
    design = np.column_stack([np.ones(len(x)), features])  # a column of ones for the intercept
    solution = np.linalg.lstsq(design, y, rcond=None)[0]
    return solution[0], solution[1:]


def predict_poly(b, w, x):
    """Predictions of the polynomial with the intercept b and the weights w."""
    return b + poly_features(x, len(w)) @ w


fig, axes = plt.subplots(1, 3, figsize=(14, 4.2), sharey=True)
for ax, degree in zip(axes, [1, 2, 15], strict=True):
    b, w = fit_poly(dist, price_m2, degree)
    ax.plot(x_grid, predict_poly(b, w, x_grid), color=INK)
    ax.scatter(dist, price_m2, s=45, color=ACCENT, edgecolor="white", linewidth=0.8, zorder=3)
    ax.set(xlabel="distance to the centre, km", ylim=(250, 2050), title=f"Degree {degree}")
axes[0].set_ylabel("price per m², $")
fig.tight_layout()
plt.show()

c2, c1, c0 = np.polyfit(dist, price_m2, 2)  # the same parabola written in km
print(f"degree 2 in km: {c0:.1f} - {-c1:.1f} x + {c2:.1f} x^2 (the truth: 1750 - 150 x + 6.2 x^2)")

Degree 1 misses the bend. The model is too simple for this data, which is called underfitting. Degree 2 has the right shape. Degree 15 has 16 parameters for 16 flats, so it passes through every point and swings between them. That is overfitting.

**Exercise 1.2.** For every degree from 1 to 15, fit the polynomial on the 16 flats and store three numbers: its RMSE on the 16 flats, its RMSE on the 200 new flats and its largest weight by absolute value.

In [ ]:
degrees = list(range(1, 16))
train_rmse, new_rmse, max_weight = [], [], []
# YOUR CODE HERE: for every degree in degrees: b, w = fit_poly(dist, price_m2, degree), then
# append rmse(...) on the 16 flats, rmse(...) on the 200 new flats (predict_poly) and max |w|

In [ ]:
assert len(train_rmse) == 15, "Fill in the loop above: 15 values in each of the three lists"
assert np.isclose(train_rmse[1], 66.7, atol=0.05), "Degree 2 should have a train RMSE of 66.7"
assert np.isclose(new_rmse[1], 74.5, atol=0.05), "Degree 2 should have an RMSE of 74.5 on new flats"
assert np.isclose(max_weight[1], 457, atol=0.5), "The largest |weight| at degree 2 should be 457"

print("degree   train RMSE   new flats RMSE   largest |weight|")
for degree in [1, 2, 3, 5, 8, 12, 15]:
    row = train_rmse[degree - 1], new_rmse[degree - 1], max_weight[degree - 1]
    print(f"{degree:6d} {row[0]:12.1f} {row[1]:16,.1f} {three_digits(row[2]):>18}")

fig, (ax_err, ax_w) = plt.subplots(1, 2, figsize=(13, 4.5))
ax_err.plot(degrees, train_rmse, "o-", ms=5, color=ACCENT, label="16 training flats")
ax_err.plot(degrees, new_rmse, "s-", ms=5, color=ORANGE, label="200 new flats")
ax_err.axhline(70, color=INK3, ls="--", lw=1.2, label="noise, 70")
ax_err.annotate("0 at degree 15", (13.8, 9), ha="right", va="bottom", fontsize=11, color=INK2)
ax_err.set(xlabel="degree", ylabel="RMSE, $", yscale="log", ylim=(8, 5e6), title="Error")
ax_err.legend(loc="upper left")
ax_w.plot(degrees, max_weight, "o-", ms=5, color=INK)
ax_w.set(xlabel="degree", ylabel="largest |weight|", yscale="log", title="Largest weight")
fig.tight_layout()
plt.show()

The training error only falls as the degree grows, down to 0.0 at degree 15. The error on new flats is U-shaped: 104.8 at degree 1, 74.5 at degree 2, then up again, and about 1.9 million dollars at degree 15. The noise has a standard deviation of 70, so no model can do much better than that on new flats, and the parabola is close. The training error cannot choose the degree, because it always votes for the most flexible model.

The weights give overfitting away. The largest one is 457 at degree 2 and 7.0 billion at degree 15. Huge weights of opposite signs cancel each other on the 16 training points and nowhere else. On the page `01_underfit_overfit.html` you can move the degree and draw new samples.

## 2. Ridge regression from scratch

Overfitting has three remedies: a simpler model, more data, or regularisation. Regularisation keeps the flexible model and adds to the loss a penalty on the size of the weights. Ridge regression penalises the sum of their squares:

$$L(\mathbf{w}, b) = \lVert y - X \mathbf{w} - b \rVert^2 + \lambda \lVert \mathbf{w} \rVert_2^2, \qquad \lVert \mathbf{w} \rVert_2^2 = \sum_j w_j^2.$$

The number $\lambda \ge 0$ is the regularisation strength, and $\lambda = 0$ is plain least squares. The intercept $b$ is not penalised: a penalty on it would pull every prediction towards zero.

### 2.1 The closed form

Subtract the column means from $X$ and the mean from $y$, and call the results $X_c$ and $y_c$. The intercept then drops out of the loss, and a zero gradient gives

$$\mathbf{w} = (X_c^\top X_c + \lambda I)^{-1} X_c^\top y_c, \qquad b = \bar y - \bar x^\top \mathbf{w}.$$

**Exercise 2.1.** Write `ridge_fit(X, y, lam)` by these formulas. Solve the system with `np.linalg.solve`, as last week, without computing the inverse.

In [ ]:
def ridge_fit(X, y, lam):
    """Ridge regression by the closed form. Returns the intercept b and the weights w."""
    # YOUR CODE HERE: centre X and y (subtract X.mean(axis=0) and y.mean()), solve
    # (Xc^T Xc + lam * I) w = Xc^T yc (np.eye, np.linalg.solve), then return the pair b, w
    return None

In [ ]:
assert poly_features(dist, 2) is not None, "Finish Exercise 1.1 first"
P15, P15_new = poly_features(dist, 15), poly_features(dist_new, 15)  # the features t, ..., t^15
ridge_answer = ridge_fit(P15, price_m2, 0.1)
assert ridge_answer is not None, "Fill in ridge_fit above"
b_ours, w_ours = ridge_answer
sk_ridge = Ridge(alpha=0.1).fit(P15, price_m2)  # scikit-learn calls lambda "alpha"
assert np.allclose(w_ours, sk_ridge.coef_, atol=1e-6), "The weights differ from sklearn's Ridge"
assert np.isclose(b_ours, sk_ridge.intercept_), "The intercept differs: b = mean(y) - mean(X) @ w"
print("the first three weights, ours:   ", w_ours[:3].round(3), " intercept:", b_ours.round(3))
print(
    "the first three weights, sklearn:",
    sk_ridge.coef_[:3].round(3),
    " intercept:",
    sk_ridge.intercept_.round(3),
)

`Ridge(alpha=lam)` from scikit-learn minimises the same loss and gives the same answer. The library calls $\lambda$ `alpha`.

### 2.2 Sweep $\lambda$ on degree 15

Take the degree 15 polynomial that failed in section 1 and fit it with different $\lambda$.

In [ ]:
assert poly_features(dist, 2) is not None, "Finish Exercise 1.1 first"
assert ridge_fit(np.eye(3), np.arange(3.0), 1.0) is not None, "Finish Exercise 2.1 first"


def ridge_curve_scores(lam):
    """Degree 15 with ridge: RMSE on the 16 flats, RMSE on the 200 new flats, largest |weight|."""
    b, w = ridge_fit(P15, price_m2, lam)
    return rmse(price_m2, b + P15 @ w), rmse(price_m2_new, b + P15_new @ w), np.abs(w).max()


print("  lambda   train RMSE   new flats RMSE   largest |weight|")
b, w = fit_poly(dist, price_m2, 15)  # lambda = 0 is least squares
row = rmse(price_m2, predict_poly(b, w, dist)), rmse(price_m2_new, predict_poly(b, w, dist_new))
print(f"{0:8g} {row[0]:12.1f} {row[1]:16,.1f} {three_digits(np.abs(w).max()):>18}")
for lam in [1e-4, 0.01, 0.1, 0.3, 1, 10, 100]:
    row = ridge_curve_scores(lam)
    print(f"{lam:8g} {row[0]:12.1f} {row[1]:16,.1f} {three_digits(row[2]):>18}")

lam_grid = np.round(np.logspace(-4, 3, 71), 6)  # 0.0001 ... 1000, ten values per decade
sweep = np.array([ridge_curve_scores(lam) for lam in lam_grid])  # columns: train, new, weight
lam_curve = lam_grid[sweep[:, 1].argmin()]  # the lambda with the lowest error on new flats
print(f"\nbest on the new flats: lambda = {lam_curve:.2f}, RMSE {sweep[:, 1].min():.1f}")

fig, axes = plt.subplots(1, 3, figsize=(14, 4.2), sharey=True)
for ax, lam in zip(axes, [0, lam_curve, 100], strict=True):
    b, w = fit_poly(dist, price_m2, 15) if lam == 0 else ridge_fit(P15, price_m2, lam)
    ax.plot(x_grid, b + poly_features(x_grid, 15) @ w, color=INK)
    ax.scatter(dist, price_m2, s=45, color=ACCENT, edgecolor="white", linewidth=0.8, zorder=3)
    ax.set(xlabel="distance to the centre, km", ylim=(250, 2050))
    ax.set_title(rf"Degree 15, $\lambda$ = {lam:.3g}")
axes[0].set_ylabel("price per m², $")
fig.tight_layout()
plt.show()

In [ ]:
assert poly_features(dist, 2) is not None, "Finish Exercise 1.1 first"
assert ridge_fit(np.eye(3), np.arange(3.0), 1.0) is not None, "Finish Exercise 2.1 first"
fig, (ax_err, ax_w) = plt.subplots(1, 2, figsize=(13, 4.5))
ax_err.plot(lam_grid, sweep[:, 0], color=ACCENT, label="16 training flats")
ax_err.plot(lam_grid, sweep[:, 1], color=ORANGE, label="200 new flats")
ax_err.scatter([lam_curve], [sweep[:, 1].min()], s=70, color=ORANGE, edgecolor="white", zorder=4)
ax_err.axhline(70, color=INK3, ls="--", lw=1.2, label="noise, 70")
ax_err.set(xscale="log", xlabel=r"$\lambda$", ylabel="RMSE, $", ylim=(0, 300), title="Error")
ax_err.legend(loc="upper left")
ax_w.plot(lam_grid, sweep[:, 2], color=INK)
ax_w.set(
    xscale="log",
    yscale="log",
    xlabel=r"$\lambda$",
    ylabel="largest |weight|",
    title="Largest weight",
)
fig.tight_layout()
plt.show()

A tiny penalty already tames the curve. At $\lambda = 0.0001$ the largest weight falls from 7.0 billion to 3,090, and the error on new flats from 1.9 million to 109.9. As $\lambda$ grows the curve calms down further and then goes flat: at $\lambda = 100$ the model is close to a constant and its RMSE on new flats is 241.0, worse than the straight line of section 1. The error on new flats is U-shaped again, this time in $\lambda$. A small $\lambda$ overfits and a large one underfits. The best value on the grid is $\lambda = 0.32$ with an RMSE of 74.3, as good as the parabola (74.5). The page `02_ridge_shrinkage.html` has the same sweep with a slider.

We picked this $\lambda$ by looking at the 200 new flats. A real project does not have them, and section 4 shows what to do then.

## 3. A listing with twins and junk

The second dataset is 40 flats from a listing site with 10 columns and a price in thousand US dollars. It is synthetic too:

$$\text{price} = 60 + 1.1 \cdot \text{area} - 3.0 \cdot \text{centre} - 4.0 \cdot \text{metro} - 0.3 \cdot \text{age} + \varepsilon, \qquad \varepsilon \sim \mathcal{N}(0, 12^2).$$

Only four columns matter. `living`, the living area, is about 0.6 of `area`, so the two are twins. `rooms` grows with `area`. `floor`, `day` (the day of the month the listing was posted), `house` (the house number) and `photos` do nothing. Another 400 flats from the same source are the test set. It stays closed until the end of section 4.

In [ ]:
FEATURES = ["area", "living", "rooms", "floor", "centre", "metro", "age", "day", "house", "photos"]
TRUE_WEIGHTS = np.array([1.1, 0, 0, 0, -3.0, -4.0, -0.3, 0, 0, 0])  # in the order of FEATURES


def flats_sample(seed, n, sd=12):
    """n flats: a matrix with the 10 columns of FEATURES and the price in thousand US dollars."""
    rng = np.random.default_rng(seed)
    area = np.round(rng.uniform(30, 120, n))  # total area, m²
    living = np.round(0.6 * area + rng.normal(0, 1.5, n), 1)  # living area, m²
    rooms = np.clip(np.round(area / 27 + rng.normal(0, 0.5, n)), 1, 5)
    floor = rng.integers(1, 13, n).astype(float)
    centre = np.round(rng.uniform(0.5, 14, n), 1)  # km to the city centre
    metro = np.round(rng.uniform(0.1, 4, n), 1)  # km to the nearest metro station
    age = rng.integers(0, 61, n).astype(float)  # age of the building, years
    day = rng.integers(1, 31, n).astype(float)
    house = rng.integers(1, 151, n).astype(float)
    photos = rng.integers(3, 26, n).astype(float)
    X = np.column_stack([area, living, rooms, floor, centre, metro, age, day, house, photos])
    noise = rng.normal(0, sd, n)
    y = np.round(60 + 1.1 * area - 3.0 * centre - 4.0 * metro - 0.3 * age + noise, 1)
    return X, y


X, y = flats_sample(2079, 40)  # the 40 flats we train on
X_test, y_test = flats_sample(7079, 400)  # the test set: 400 new flats, closed until section 4.5

corr = np.corrcoef(X, rowvar=False)
print(f"correlation of area and living: {corr[0, 1]:.3f}")
print(f"correlation of area and rooms:  {corr[0, 2]:.2f}")
pd.DataFrame(X, columns=FEATURES).assign(price=y).head()

### 3.1 Least squares

Fit plain least squares on all 10 columns and put the weights next to the truth.

In [ ]:
ols = LinearRegression().fit(X, y)
weights = pd.DataFrame({"truth": TRUE_WEIGHTS, "least squares": ols.coef_}, index=FEATURES)
print(f"RMSE on the 40 flats: {rmse(y, ols.predict(X)):.1f}")
weights.round(2)

The weights of `centre` and `metro` are close to the truth. The weights of the twins are far from it: `area` gets 2.43 where the truth is 1.1, `living` gets $-1.72$ where the truth is 0, and `rooms` gets $-5.55$. Read literally, every extra m² of living area makes a flat 1.72 thousand dollars cheaper. The correlation between `area` and `living` is 0.995, so the data cannot tell how to share the weight between them. Last week two kinds of cholesterol did the same, and the remedy was left for later.

### 3.2 Ridge shares the weight

Ridge adds $\lambda I$ to $X^\top X$, and the sum is invertible however similar the columns are. We standardise the features first (section 3.4 shows why) and take $\lambda = 2.5119$ from a grid. Section 4 shows where this value comes from.

In [ ]:
RIDGE_GRID = np.round(np.logspace(-2, 3, 51), 4)  # 0.01 ... 1000, ten values per decade
lam_demo = RIDGE_GRID[24]  # 2.5119

scaler = StandardScaler().fit(X)
Z = scaler.transform(X)  # every column now has mean 0 and standard deviation 1
ridge = Ridge(alpha=lam_demo).fit(Z, y)
# a weight per standard deviation, divided by the standard deviation, is a weight per unit
weights["ridge"] = ridge.coef_ / scaler.scale_
per_m2 = weights.loc["area", "ridge"] + 0.6 * weights.loc["living", "ridge"]
print(f"area + 0.6 * living under ridge: {per_m2:.2f} thousand dollars per m²")

ridge_path = np.array([Ridge(alpha=lam).fit(Z, y).coef_ for lam in RIDGE_GRID])  # shape (51, 10)
fig, ax = plt.subplots(figsize=(9, 4.8))
for j, name in enumerate(FEATURES):
    twin = name in ("area", "living")
    color = ACCENT if name == "area" else ORANGE if name == "living" else INK3
    ax.plot(RIDGE_GRID, ridge_path[:, j], color=color, lw=2.2 if twin else 1.2)
    if twin:
        ax.annotate(
            name,
            (RIDGE_GRID[0], ridge_path[0, j]),
            xytext=(6, 7),
            textcoords="offset points",
            color=color,
        )
ax.axhline(0, color=LINE, lw=0.8)
ax.axvline(lam_demo, color=INK3, ls="--", lw=1.2)
ax.set(
    xscale="log",
    xlabel=r"$\lambda$",
    ylabel="weight of the standardised feature",
    title="Ridge: the weights against $\\lambda$",
)
plt.show()

weights.round(2)

Under ridge `area` gets 0.71 and `living` gets 0.84: the twins share the weight. A flat with one more m² of total area has about 0.6 m² more living area, so the pair adds $0.71 + 0.6 \times 0.84 \approx 1.2$ thousand dollars per m², close to the true 1.1. On the plot the two lines start far apart, come together, and then shrink with the rest. Ridge sets no weight exactly to zero, so the junk columns keep small weights.

Last week the weights of correlated features jumped from one resample to the next. Refit both models on 200 bootstrap resamples of the 40 flats (40 rows drawn with replacement) and look at the twins.

In [ ]:
rng = np.random.default_rng(SEED)
boot_ols, boot_ridge = [], []
for _ in range(200):
    rows = rng.integers(0, len(y), size=len(y))  # 40 row numbers, with repeats
    boot_ols.append(LinearRegression().fit(X[rows], y[rows]).coef_[:2])
    model = make_pipeline(StandardScaler(), Ridge(alpha=lam_demo)).fit(X[rows], y[rows])
    boot_ridge.append(model[-1].coef_[:2] / model[0].scale_[:2])  # back to weights per m²
boot_ols, boot_ridge = np.array(boot_ols), np.array(boot_ridge)  # columns: area, living

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(*boot_ols.T, s=14, color=ORANGE, alpha=0.6, linewidth=0, label="least squares")
ax.scatter(*boot_ridge.T, s=14, color=ACCENT, alpha=0.6, linewidth=0, label="ridge")
ax.scatter([1.1], [0], s=110, marker="*", color=INK, zorder=5, label="truth")
ax.set(
    xlabel="weight of area", ylabel="weight of living", title="The twins in 200 bootstrap refits"
)
ax.legend(loc="upper right")
plt.show()

boot = pd.DataFrame(
    {
        "least squares, area": boot_ols[:, 0],
        "least squares, living": boot_ols[:, 1],
        "ridge, area": boot_ridge[:, 0],
        "ridge, living": boot_ridge[:, 1],
    }
)
boot.agg(["min", "max", "std"]).T.round(2)

Under least squares the weight of `area` runs from $-1.7$ to 6.1 across the resamples and the weight of `living` moves against it, along a line. Under ridge the twins stop swinging: the weight of `area` stays between 0.5 and 1.1.

### 3.3 Lasso drops features

Lasso penalises the sum of absolute values, $\lVert \mathbf{w} \rVert_1 = \sum_j \lvert w_j \rvert$, in place of the sum of squares. It has no closed form, and scikit-learn fits it by coordinate descent. Unlike ridge, it sets some weights exactly to zero.

scikit-learn scales the two losses differently: `Ridge(alpha)` minimises $\lVert y - X \mathbf{w} \rVert^2 + \alpha \lVert \mathbf{w} \rVert_2^2$ and `Lasso(alpha)` minimises $\frac{1}{2n} \lVert y - X \mathbf{w} \rVert^2 + \alpha \lVert \mathbf{w} \rVert_1$. So a ridge $\lambda$ and a lasso $\lambda$ cannot be compared with each other.

**Exercise 3.1.** For every `lam` in `LASSO_GRID`, fit `Lasso(alpha=lam, max_iter=100_000)` on the standardised features `Z` and append its weights (`coef_`) to `lasso_path`.

In [ ]:
LASSO_GRID = np.round(np.logspace(-2, 1.5, 36), 4)  # 0.01 ... 31.6, ten values per decade
lasso_path = []
# YOUR CODE HERE: for every lam in LASSO_GRID: fit Lasso(alpha=lam, max_iter=100_000) on Z, y
# and append its coef_ to lasso_path
lasso_path = np.array(lasso_path)  # shape (36, 10): one row of weights per lambda

In [ ]:
assert lasso_path.shape == (36, 10), "Fill in the loop above: one row of 10 weights per lambda"
n_left = (lasso_path != 0).sum(axis=1)  # how many weights are not zero, for every lambda
assert n_left[0] == 10 and n_left[-1] == 0, "Expect 10 weights at lambda = 0.01 and none at 31.6"

for lam in [0.01, 0.5012, 1.5849, 3.1623, 10.0]:
    i = int(np.argmin(np.abs(LASSO_GRID - lam)))
    left = [name for name, weight in zip(FEATURES, lasso_path[i], strict=True) if weight != 0]
    print(f"lambda = {LASSO_GRID[i]:7.4f}: {len(left):2d} left: {', '.join(left)}")
weights["lasso"] = lasso_path[np.argmin(np.abs(LASSO_GRID - 1.5849))] / scaler.scale_

fig, (ax_path, ax_count) = plt.subplots(1, 2, figsize=(13, 4.8), width_ratios=[3, 2])
groups = {"area, centre, metro, age": ACCENT, "living, rooms": ORANGE, "the other four": INK3}
for j, name in enumerate(FEATURES):
    group = next((key for key in groups if name in key.split(", ")), "the other four")
    ax_path.plot(LASSO_GRID, lasso_path[:, j], color=groups[group], lw=1.8)
for group, color in groups.items():
    ax_path.plot([], [], color=color, label=group)
ax_path.axhline(0, color=LINE, lw=0.8)
ax_path.set(
    xscale="log",
    xlabel=r"$\lambda$",
    ylabel="weight of the standardised feature",
    title="Lasso: the weights against $\\lambda$",
)
ax_path.legend(loc="upper right")
ax_count.step(LASSO_GRID, n_left, where="mid", color=INK)
ax_count.set(
    xscale="log", xlabel=r"$\lambda$", ylabel="weights that are not zero", title="Features left"
)
fig.tight_layout()
plt.show()

weights.round(2)

At $\lambda = 0.01$ all 10 weights are alive. As $\lambda$ grows they hit zero one after another: 8 are left at $\lambda = 0.5$, 6 at 1.58, 4 at 3.16 and 2 at 10. At $\lambda = 1.58$ the column `living` is gone and `area` carries the whole effect with a weight of 1.16 per m². At $\lambda = 3.16$ exactly the four true features are left. Lasso selects features while it fits the model. The page `03_l1_l2_geometry.html` shows why the L1 penalty gives exact zeros, and `04_lasso_feature_selection.html` has this path with a slider.

### 3.4 Why standardise first

The penalty adds up the weights of all features, and the size of a weight depends on the units of its feature. Fit ridge with the same $\lambda$ five times: on raw features with both areas in m², in cm² and in km², and on standardised features with the areas in m² and in km².

In [ ]:
def ridge_in_units(factor, standardise):
    """Ridge with lam_demo when both areas are multiplied by factor (1e4: cm², 1e-6: km²)."""
    X_units = X.copy()
    X_units[:, :2] *= factor
    if standardise:
        model = make_pipeline(StandardScaler(), Ridge(alpha=lam_demo)).fit(X_units, y)
        coef = model[-1].coef_ / model[0].scale_
    else:
        model = Ridge(alpha=lam_demo).fit(X_units, y)
        coef = model.coef_
    return {
        "area, per m²": coef[0] * factor,
        "living, per m²": coef[1] * factor,
        "rooms": coef[2],
        "RMSE on the 40 flats": rmse(y, model.predict(X_units)),
    }


units = pd.DataFrame(
    {
        "raw, areas in m²": ridge_in_units(1, standardise=False),
        "raw, areas in cm²": ridge_in_units(1e4, standardise=False),
        "raw, areas in km²": ridge_in_units(1e-6, standardise=False),
        "standardised, areas in m²": ridge_in_units(1, standardise=True),
        "standardised, areas in km²": ridge_in_units(1e-6, standardise=True),
    }
).T
units.round(2)

In m² and in cm² the areas are large numbers, so their weights are small and the penalty hardly touches them. The twins look almost as they did under least squares: 2.37 and $-1.68$ in m². In km² the same areas are tiny numbers that need weights in the millions. The penalty forbids such weights, and the model loses the area: its RMSE on the 40 flats rises from 10.6 to 19.7, and `rooms` takes over with a weight of 17.8. The flats are the same in all three rows, only the units differ.

After `StandardScaler` every column has a standard deviation of 1, and the last two rows are identical. The same holds for lasso. In week 1 kNN needed scaling for the same reason: the units of a feature decided how much it counted.

## 4. Choosing $\lambda$ honestly

The weights are parameters: the model learns them from the training data. $\lambda$ is a hyperparameter: we set it before training, like the degree in section 1 or $k$ in kNN. The training error cannot choose it, because the training error is smallest at $\lambda = 0$. The test set cannot either, because it is kept for one final number. That leaves the 40 training flats.

### 4.1 One split is noisy

The simplest idea is the one from week 1: hold out 8 of the 40 flats for validation and train on the other 32. Do it 200 times with different random splits, each time for plain least squares.

In [ ]:
split_rmse = []
for seed in range(200):
    order = np.random.default_rng(seed).permutation(len(y))
    val_idx, train_idx = order[:8], order[8:]  # 8 flats for validation, 32 for training
    model = LinearRegression().fit(X[train_idx], y[train_idx])
    split_rmse.append(rmse(y[val_idx], model.predict(X[val_idx])))
split_rmse = np.array(split_rmse)
print(
    f"validation RMSE over 200 splits: from {split_rmse.min():.1f} to {split_rmse.max():.1f}, "
    f"mean {split_rmse.mean():.1f}, standard deviation {split_rmse.std():.1f}"
)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(split_rmse, bins=20, color=ACCENT, alpha=0.4, edgecolor="white")
ax.set(
    xlabel="validation RMSE of least squares, thousand $",
    ylabel="splits out of 200",
    title="One model, 200 random 32/8 splits",
)
plt.show()

The same model on the same 40 flats gets a validation RMSE anywhere from 5.5 to 24.7, depending on which 8 flats were held out. The mean is 14.5 and the standard deviation is 3.4. The models we want to compare differ by less than that, so one split of 40 flats cannot rank them.

### 4.2 k-fold cross-validation by hand

Cut the 40 flats into $k = 5$ folds of 8. Each fold is the validation set once while the model trains on the other four, so every flat is validated exactly once. Then average the five errors.

In [ ]:
kfold = KFold(n_splits=5, shuffle=True, random_state=0)
for number, (train_idx, val_idx) in enumerate(kfold.split(X), start=1):
    print(f"fold {number}: train on {len(train_idx)} flats, validate on flats {val_idx}")

Everything that is fitted from data belongs inside the loop. That includes the scaler: it may see only the training folds.

**Exercise 4.1.** Write `kfold_mse(model, X, y, kfold)`. For every split, fit a `StandardScaler` on the training rows, fit the model on the scaled training rows, and append its MSE on the scaled validation rows.

In [ ]:
def kfold_mse(model, X, y, kfold):
    """MSE of the model on each validation fold. The scaler sees only the training folds."""
    fold_mse = []
    # YOUR CODE HERE: for train_idx, val_idx in kfold.split(X): fit StandardScaler on X[train_idx];
    # fit the model on the scaled training rows; predict the scaled validation rows;
    # append the mean squared error on the validation rows to fold_mse
    return np.array(fold_mse)

In [ ]:
ols_folds = kfold_mse(LinearRegression(), X, y, kfold)
assert len(ols_folds) == 5, "Fill in kfold_mse above: one MSE per fold"
assert np.allclose(np.sqrt(ols_folds), [15.7, 15.4, 17.5, 12.9, 13.9], atol=0.06), (
    "Expect the fold RMSE 15.7, 15.4, 17.5, 12.9, 13.9: scaler and model are fitted on the "
    "training rows, the error is measured on the validation rows"
)
ols_cv_rmse = np.sqrt(ols_folds.mean())  # average the MSE over the folds, then take the root
print("RMSE on each fold:    ", np.sqrt(ols_folds).round(1))
print(f"cross-validation RMSE: {ols_cv_rmse:.1f}")
print(f"training RMSE:         {rmse(y, ols.predict(X)):.1f}")

The folds give between 12.9 and 17.5, and together a cross-validation RMSE of 15.2. On the flats it was trained on, the same model showed 10.6.

Now run the same function for every $\lambda$ on a grid, once for ridge and once for lasso.

In [ ]:
assert len(kfold_mse(LinearRegression(), X, y, kfold)) == 5, "Finish Exercise 4.1 first"
ridge_folds = np.array([kfold_mse(Ridge(alpha=lam), X, y, kfold) for lam in RIDGE_GRID])
lasso_folds = np.array(
    [kfold_mse(Lasso(alpha=lam, max_iter=100_000), X, y, kfold) for lam in LASSO_GRID]
)  # shape (36, 5): one row per lambda, one column per fold
ridge_cv_rmse = np.sqrt(ridge_folds.mean(axis=1))
lasso_cv_rmse = np.sqrt(lasso_folds.mean(axis=1))
lam_ridge = RIDGE_GRID[ridge_cv_rmse.argmin()]
lam_lasso = LASSO_GRID[lasso_cv_rmse.argmin()]
print(f"ridge: lowest cross-validation RMSE {ridge_cv_rmse.min():.1f} at lambda = {lam_ridge}")
print(f"lasso: lowest cross-validation RMSE {lasso_cv_rmse.min():.1f} at lambda = {lam_lasso}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
curves = [
    ("Ridge", RIDGE_GRID, ridge_cv_rmse, lam_ridge),
    ("Lasso", LASSO_GRID, lasso_cv_rmse, lam_lasso),
]
for ax, (name, grid, cv_rmse, lam_best) in zip(axes, curves, strict=True):
    ax.plot(grid, cv_rmse, color=ORANGE, label="5-fold cross-validation")
    ax.scatter([lam_best], [cv_rmse.min()], s=70, color=ORANGE, edgecolor="white", zorder=4)
    ax.axhline(ols_cv_rmse, color=INK3, ls="--", lw=1.2, label="least squares")
    ax.set(xscale="log", xlabel=r"$\lambda$", ylim=(12, 26), title=name)
axes[0].set_ylabel("cross-validation RMSE, thousand $")
axes[0].legend(loc="upper left")
fig.tight_layout()
plt.show()

Both curves have a minimum inside the grid. Ridge is best at $\lambda = 2.51$ with a cross-validation RMSE of 14.4, and lasso at $\lambda = 1.58$ with 13.4. Both beat least squares (15.2), which is where each curve starts on the left. The two $\lambda$ axes use different scales, as section 3.3 explained. The page `05_cross_validation.html` steps through the folds one at a time.

### 4.3 The same with scikit-learn

A pipeline joins the scaler and the model into one estimator, so the scaler is refitted inside every fold without any effort from us. `GridSearchCV` runs the loop over the folds and the grid, and then refits the best pipeline on all the data. scikit-learn maximises scores, so the error goes in with a minus sign: `scoring="neg_mean_squared_error"`.

**Exercise 4.2.** Build `ridge_search`: a `GridSearchCV` over the pipeline `make_pipeline(StandardScaler(), Ridge())` with the grid `{"ridge__alpha": RIDGE_GRID}`, `cv=kfold` and the scoring above. Fit it on `X`, `y`.

In [ ]:
# YOUR CODE HERE: ridge_search = GridSearchCV(pipeline, grid, cv=..., scoring=...).fit(X, y)
ridge_search = None

In [ ]:
assert ridge_search is not None, "Fill in ridge_search above"
assert hasattr(ridge_search, "best_params_"), "Fill in ridge_search above: call .fit(X, y) on it"
best_alpha = ridge_search.best_params_["ridge__alpha"]
search_rmse = np.sqrt(-ridge_search.best_score_)
assert np.isclose(
    best_alpha, 2.5119
), "Expect the best alpha 2.5119: check the grid, cv and scoring"
print(f"GridSearchCV: best alpha = {best_alpha}, cross-validation RMSE {search_rmse:.1f}")

`GridSearchCV` finds the same $\lambda$ and the same error as the loop written by hand. For lasso only the last step of the pipeline changes.

`RidgeCV` and `LassoCV` are shortcuts that search over $\lambda$ inside one `fit`. In the cell below they stand after the scaler in a pipeline, so the scaler is fitted once on all 40 flats, before the folds are cut. That is a small leak. Here it does not change the chosen $\lambda$. With heavier preprocessing it can, and a search over the whole pipeline is the safe habit.

In [ ]:
lasso_search = GridSearchCV(
    make_pipeline(StandardScaler(), Lasso(max_iter=100_000)),
    {"lasso__alpha": LASSO_GRID},
    cv=kfold,
    scoring="neg_mean_squared_error",
).fit(X, y)
print(
    f"GridSearchCV, lasso: best alpha = {lasso_search.best_params_['lasso__alpha']}, "
    f"cross-validation RMSE {np.sqrt(-lasso_search.best_score_):.1f}"
)

ridge_cv = RidgeCV(alphas=RIDGE_GRID, cv=kfold, scoring="neg_mean_squared_error")
lasso_cv = LassoCV(alphas=LASSO_GRID, cv=kfold, max_iter=100_000)
ridge_shortcut = make_pipeline(StandardScaler(), ridge_cv).fit(X, y)
lasso_shortcut = make_pipeline(StandardScaler(), lasso_cv).fit(X, y)
print(f"RidgeCV: alpha = {ridge_shortcut[-1].alpha_}")
print(f"LassoCV: alpha = {lasso_shortcut[-1].alpha_}")

### 4.4 The one-standard-error rule

The minimum of the curve is a noisy estimate too, since the five folds disagree with each other. The one-standard-error rule takes the largest $\lambda$ whose error is within one standard error of the minimum. That is the simplest model which is about as good as the best one.

In [ ]:
assert len(kfold_mse(LinearRegression(), X, y, kfold)) == 5, "Finish Exercise 4.1 first"
lasso_mean = lasso_folds.mean(axis=1)  # mean MSE over the folds, for every lambda
lasso_se = lasso_folds.std(axis=1, ddof=1) / np.sqrt(5)  # standard error of that mean
i_min = lasso_mean.argmin()
threshold = lasso_mean[i_min] + lasso_se[i_min]
lam_lasso_1se = LASSO_GRID[lasso_mean <= threshold].max()

for label, lam in [("minimum", lam_lasso), ("one standard error", lam_lasso_1se)]:
    coef = Lasso(alpha=lam, max_iter=100_000).fit(Z, y).coef_
    left = [name for name, weight in zip(FEATURES, coef, strict=True) if weight != 0]
    print(f"{label}: lambda = {lam}, {len(left)} features: {', '.join(left)}")

fig, ax = plt.subplots(figsize=(8.5, 4.5))
ax.fill_between(
    LASSO_GRID,
    np.sqrt(lasso_mean - lasso_se),
    np.sqrt(lasso_mean + lasso_se),
    color=ORANGE,
    alpha=0.18,
    linewidth=0,
    label="one standard error",
)
ax.plot(LASSO_GRID, lasso_cv_rmse, color=ORANGE, label="5-fold cross-validation")
ax.axhline(np.sqrt(threshold), color=INK3, ls="--", lw=1.2, label="minimum plus one standard error")
marks = [
    (lam_lasso, "minimum", (0, -20), "center"),
    (lam_lasso_1se, "the rule's choice", (10, -14), "left"),
]
for lam, text, offset, align in marks:
    height = lasso_cv_rmse[LASSO_GRID == lam][0]
    ax.scatter([lam], [height], s=70, color=INK, edgecolor="white", zorder=4)
    ax.annotate(
        text,
        (lam, height),
        xytext=offset,
        textcoords="offset points",
        ha=align,
        fontsize=11,
        color=INK2,
    )
ax.set(
    xscale="log",
    xlabel=r"$\lambda$",
    ylabel="cross-validation RMSE, thousand $",
    xlim=(0.05, 12),
    ylim=(10.5, 20),
    title="Lasso",
)
ax.legend(loc="upper left")
plt.show()

For lasso the rule moves $\lambda$ from 1.58 to 3.16. The model at the minimum keeps 6 features, and two of them are junk (`day` and `house`). The model at 3.16 keeps exactly the four true ones.

### 4.5 The test set, once

Every choice is made, so the 400 test flats can be opened. Refit each model on all 40 flats and compare. The last row is least squares on the four columns that matter, a model we can build only because we know how the data was made.

In [ ]:
assert len(kfold_mse(LinearRegression(), X, y, kfold)) == 5, "Finish Exercise 4.1 first"
true_four = [FEATURES.index(name) for name in ("area", "centre", "metro", "age")]


def final_scores(model, columns=None):
    """RMSE by cross-validation, then after a refit on all 40 flats: on them and on the test set."""
    columns = list(range(X.shape[1])) if columns is None else columns
    cv_rmse = np.sqrt(kfold_mse(model, X[:, columns], y, kfold).mean())
    fitted = make_pipeline(StandardScaler(), model).fit(X[:, columns], y)
    return {
        "5-fold cross-validation": cv_rmse,
        "40 training flats": rmse(y, fitted.predict(X[:, columns])),
        "400 test flats": rmse(y_test, fitted.predict(X_test[:, columns])),
    }


results = pd.DataFrame(
    {
        "baseline: the mean price": final_scores(DummyRegressor()),
        "least squares, 10 columns": final_scores(LinearRegression()),
        f"ridge, lambda = {lam_ridge:.2f}": final_scores(Ridge(alpha=lam_ridge)),
        f"lasso, lambda = {lam_lasso:.2f} (minimum)": final_scores(
            Lasso(alpha=lam_lasso, max_iter=100_000)
        ),
        f"lasso, lambda = {lam_lasso_1se:.2f} (one standard error)": final_scores(
            Lasso(alpha=lam_lasso_1se, max_iter=100_000)
        ),
        "least squares, the 4 true columns": final_scores(LinearRegression(), true_four),
    }
).T
results.round(1)

The order that cross-validation gave on 40 flats holds on the 400 new ones: least squares 14.5, ridge 13.4, lasso 13.1. On the 40 training flats the order is the reverse, and least squares has the smallest error, 10.6. Lasso at the minimum is within 0.1 of least squares on the four true columns (13.0). The one-standard-error model is simpler and pays a little for it (13.6), because its weights are shrunk more. Every model is far ahead of the baseline's 34.8.

## 5. Real data, and a split that leaks

### 5.1 Diabetes with 65 features

Back to the 442 diabetes patients of last week, with the same 89 patients set aside as the test set. Last week linear regression on the 10 features reached a test RMSE of 53.9. A more flexible model might do better: `PolynomialFeatures(degree=2)` adds every square and every pairwise product, which makes 65 features out of 10. The other 353 patients are for training and cross-validation.

In [ ]:
diabetes = load_diabetes(as_frame=True, scaled=False)  # bundled with scikit-learn, no download
patients_train, patients_test, target_train, target_test = train_test_split(
    diabetes.data, diabetes.target, test_size=0.2, random_state=SEED
)


def train_and_cv(model):
    """RMSE on the training patients and by 5-fold cross-validation. Leaves the model fitted."""
    scores = cross_val_score(
        model, patients_train, target_train, cv=kfold, scoring="neg_mean_squared_error"
    )
    model.fit(patients_train, target_train)
    return {
        "train RMSE": rmse(target_train, model.predict(patients_train)),
        "cross-validation RMSE": np.sqrt(-scores.mean()),
    }


linear_10 = make_pipeline(StandardScaler(), LinearRegression())
linear_65 = make_pipeline(
    PolynomialFeatures(degree=2, include_bias=False), StandardScaler(), LinearRegression()
)
diabetes_models = {
    "baseline: the mean": DummyRegressor(),
    "least squares, 10 features": linear_10,
    "least squares, 65 features": linear_65,
}
diabetes_scores = {name: train_and_cv(model) for name, model in diabetes_models.items()}
n_poly = linear_65[-1].coef_.shape[0]
print(
    f"{len(patients_train)} patients for training, {len(patients_test)} for the test, "
    f"{n_poly} features after PolynomialFeatures"
)
pd.DataFrame(diabetes_scores).T.round(1)

With 65 features least squares fits the training patients better (48.9 against 53.6) and predicts new ones worse: its cross-validation RMSE is 61.4 against 55.0. This is section 1 again, on real data. The 65 features also contain perfect twins: `sex` takes two values, so `sex^2` carries exactly the same information.

**Exercise 5.1.** Build `poly_ridge_search`: a `GridSearchCV` over the pipeline `PolynomialFeatures(degree=2, include_bias=False)`, `StandardScaler()`, `Ridge()` with the grid `{"ridge__alpha": WIDE_GRID}`, `cv=kfold` and `scoring="neg_mean_squared_error"`. Fit it on the training patients.

In [ ]:
WIDE_GRID = np.round(np.logspace(-2, 4, 31), 4)  # 0.01 ... 10,000, five values per decade
# YOUR CODE HERE: the search as in Exercise 4.2, with PolynomialFeatures as the first step of
# the pipeline; fit it on patients_train, target_train
poly_ridge_search = None

In [ ]:
assert poly_ridge_search is not None, "Fill in poly_ridge_search above"
assert hasattr(
    poly_ridge_search, "best_params_"
), "Fill in poly_ridge_search above: call .fit on it"
lam_diabetes = poly_ridge_search.best_params_["ridge__alpha"]
assert np.isclose(lam_diabetes, 39.8107), "Expect the best alpha 39.8107: check the pipeline and cv"
poly_ridge = poly_ridge_search.best_estimator_  # already refitted on all training patients
diabetes_models[f"ridge, 65 features, lambda = {lam_diabetes:.1f}"] = poly_ridge
diabetes_scores[f"ridge, 65 features, lambda = {lam_diabetes:.1f}"] = {
    "train RMSE": rmse(target_train, poly_ridge.predict(patients_train)),
    "cross-validation RMSE": np.sqrt(-poly_ridge_search.best_score_),
}

# the same search for lasso
poly_lasso_search = GridSearchCV(
    make_pipeline(
        PolynomialFeatures(degree=2, include_bias=False), StandardScaler(), Lasso(max_iter=100_000)
    ),
    {"lasso__alpha": np.round(np.logspace(-1, 1.5, 26), 4)},  # 0.1 ... 31.6
    cv=kfold,
    scoring="neg_mean_squared_error",
).fit(patients_train, target_train)
lam_diabetes_lasso = poly_lasso_search.best_params_["lasso__alpha"]
poly_lasso = poly_lasso_search.best_estimator_
n_lasso_left = int((poly_lasso[-1].coef_ != 0).sum())
diabetes_models[f"lasso, 65 features, lambda = {lam_diabetes_lasso:.2f}"] = poly_lasso
diabetes_scores[f"lasso, 65 features, lambda = {lam_diabetes_lasso:.2f}"] = {
    "train RMSE": rmse(target_train, poly_lasso.predict(patients_train)),
    "cross-validation RMSE": np.sqrt(-poly_lasso_search.best_score_),
}
print(f"lasso keeps {n_lasso_left} of the {n_poly} features")

ridge_curve = np.sqrt(-poly_ridge_search.cv_results_["mean_test_score"])
fig, ax = plt.subplots(figsize=(8.5, 4.5))
ax.plot(WIDE_GRID, ridge_curve, color=ORANGE, label="ridge, 65 features")
ax.scatter([lam_diabetes], [ridge_curve.min()], s=70, color=ORANGE, edgecolor="white", zorder=4)
ax.axhline(
    diabetes_scores["least squares, 65 features"]["cross-validation RMSE"],
    color=INK,
    ls=":",
    lw=1.6,
    label="least squares, 65 features",
)
ax.axhline(
    diabetes_scores["least squares, 10 features"]["cross-validation RMSE"],
    color=INK3,
    ls="--",
    lw=1.2,
    label="least squares, 10 features",
)
ax.set(xscale="log", xlabel=r"$\lambda$", ylabel="cross-validation RMSE", ylim=(53, 70))
ax.legend(loc="upper left")
plt.show()

pd.DataFrame(diabetes_scores).T.round(1)

Ridge with $\lambda = 39.8$ brings the 65 features from 61.4 down to 54.8. Lasso reaches 55.0 and keeps 11 of the 65 features. Both are level with plain least squares on the 10 original features (55.0), and the gaps between the three are much smaller than the spread between folds. The extra features carry little signal on this data. Regularisation stops them from doing harm, and the simple model remains a fair choice.

Nothing else will be tuned, so open the test set.

In [ ]:
assert poly_ridge_search is not None, "Finish Exercise 5.1 first"
for name, model in diabetes_models.items():
    diabetes_scores[name]["test RMSE"] = rmse(target_test, model.predict(patients_test))
pd.DataFrame(diabetes_scores).T.round(1)

The test set agrees with cross-validation. Least squares on 10 features gets 53.9, the number from last week. On 65 features it gets 55.6 without regularisation, 53.3 with ridge and 53.4 with lasso. The baseline is at 73.2.

### 5.2 Buildings: a split that leaks

The last dataset has 30 buildings with 4 listed flats in each. Flats of one building share a hidden building effect on the price (the yard, the renovation, the view):

$$\text{price} = 60 + 1.1 \cdot \text{area} - 3.0 \cdot \text{centre} + \text{building effect} + \varepsilon, \qquad \text{building effect} \sim \mathcal{N}(0, 15^2), \quad \varepsilon \sim \mathcal{N}(0, 5^2).$$

The model is ridge with $\lambda = 1$ on the area, the distance to the centre and the building as a categorical feature, one-hot encoded as in week 1. First a random 5-fold split, as before.

In [ ]:
def buildings_sample(seed, n_buildings, per_building):
    """Flats that share a building share a hidden effect on the price, N(0, 15) thousand dollars."""
    rng = np.random.default_rng(seed)
    effect = rng.normal(0, 15, n_buildings)
    centre = np.round(rng.uniform(0.5, 14, n_buildings), 1)  # one distance per building
    building = np.repeat(np.arange(n_buildings), per_building)
    area = np.round(rng.uniform(30, 120, n_buildings * per_building))
    noise = rng.normal(0, 5, n_buildings * per_building)
    price = np.round(60 + 1.1 * area - 3.0 * centre[building] + effect[building] + noise, 1)
    return pd.DataFrame(
        {"building": building, "area": area, "centre": centre[building], "price": price}
    )


houses = buildings_sample(31, 30, 4)  # 30 buildings, 4 flats in each
new_houses = buildings_sample(3131, 300, 1)  # 300 other buildings, one flat in each
new_houses["building"] += 30  # none of them is one of the 30 buildings above
house_columns = ["building", "area", "centre"]

# handle_unknown="ignore": a building the encoder has not seen gets zeros in every column
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
with_building = make_pipeline(
    make_column_transformer((StandardScaler(), ["area", "centre"]), (encoder, ["building"])),
    Ridge(alpha=1.0),
)
random_mse = -cross_val_score(
    with_building,
    houses[house_columns],
    houses["price"],
    cv=kfold,
    scoring="neg_mean_squared_error",
)
print(f"random 5-fold cross-validation: RMSE {np.sqrt(random_mse.mean()):.1f}")
houses.head(6)

**Exercise 5.2.** Compute `grouped_mse`: the same call with `cv=GroupKFold(n_splits=5)` and `groups=houses["building"]`. This split keeps all flats of a building on one side.

In [ ]:
# YOUR CODE HERE: the cross_val_score call from the cell above with the new cv and with groups.
# Keep the minus sign: scikit-learn returns the negative MSE
grouped_mse = None

In [ ]:
assert grouped_mse is not None, "Fill in grouped_mse above"
assert np.isclose(
    np.sqrt(np.mean(grouped_mse)), 13.2, atol=0.05
), "Expect an RMSE of 13.2: cv=GroupKFold(n_splits=5), groups=houses['building'], minus sign"
print(f"grouped 5-fold cross-validation: RMSE {np.sqrt(grouped_mse.mean()):.1f}")

The two splits disagree by a factor of almost two. To see which one is right, fit the model on all 120 flats and predict flats in 300 buildings it has never seen. For comparison, do everything again for a model without the building column.

In [ ]:
assert grouped_mse is not None, "Finish Exercise 5.2 first"
without_building = make_pipeline(
    make_column_transformer((StandardScaler(), ["area", "centre"])), Ridge(alpha=1.0)
)


def three_numbers(model):
    """RMSE by a random split, by a grouped split, and on 300 new buildings."""
    scores = {}
    splits = [
        ("random 5-fold", kfold, None),
        ("grouped 5-fold", GroupKFold(n_splits=5), houses["building"]),
    ]
    for name, cv, groups in splits:
        mse = -cross_val_score(
            model,
            houses[house_columns],
            houses["price"],
            cv=cv,
            groups=groups,
            scoring="neg_mean_squared_error",
        )
        scores[name] = np.sqrt(mse.mean())
    model.fit(houses[house_columns], houses["price"])
    scores["300 new buildings"] = rmse(
        new_houses["price"], model.predict(new_houses[house_columns])
    )
    return scores


leak = pd.DataFrame(
    {
        "with the building column": three_numbers(with_building),
        "without the building column": three_numbers(without_building),
    }
).T

seen = [
    np.isin(houses["building"].iloc[val_idx], houses["building"].iloc[train_idx]).mean()
    for train_idx, val_idx in kfold.split(houses)
]
print(f"random split: {np.mean(seen):.0%} of the validation flats have their building in training")

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(
    leak.columns, leak.loc["with the building column"], width=0.55, color=[ORANGE, ACCENT, PURPLE]
)
ax.bar_label(bars, fmt="%.1f", padding=4, color=INK2)
ax.set(ylabel="RMSE, thousand $", ylim=(0, 18), title="The model with the building column")
plt.show()

leak.round(1)

The random split promises an RMSE of 7.0, and on 300 new buildings the same model gets 15.5. In the random split every validation flat has neighbours from its own building in the training folds, so the model has already learned that building's effect. A new building has no column of its own, and the error is back at the size of the building effect. The grouped split says 13.2, which is much closer.

The leak also picks the wrong model. Under the random split the building column looks like a large improvement, 7.0 against 13.5 without it. Under the grouped split and on new buildings it changes almost nothing. A validation set has to look like the data the model will meet, and here that means flats in buildings it has never seen. The page `06_splits_that_leak.html` shows both splits, and a split by time.

## 6. What to remember

- The training error only falls as a model gets more flexible, while the error on new data is U-shaped. Huge weights are a sign of overfitting.
- Ridge adds $\lambda \lVert \mathbf{w} \rVert_2^2$ to the loss and has a closed form. It shrinks every weight, shares the weight between correlated features and sets none to zero. Lasso adds $\lambda \lVert \mathbf{w} \rVert_1$ and sets some weights exactly to zero.
- Standardise the features before regularising, and leave the intercept out of the penalty.
- $\lambda$ is a hyperparameter. Choose it by cross-validation on the training data with the scaler inside the folds, which is what a pipeline inside `GridSearchCV` does. Open the test set once.
- Split the data the way the model will be used. When objects come in groups, whole groups go to one side.

## Try at home

1. In section 1, train on 200 flats in place of 16 (`curve_sample(68, 200)`) and repeat the loop over the degrees. What happens to the error of degree 15 on new flats?
2. `ElasticNet(alpha, l1_ratio)` from `sklearn.linear_model` mixes the two penalties. Search over both of its parameters on the listing with `GridSearchCV`. Does the best model keep both twins or one of them?
3. Cross-validation depends on how the folds are cut. Repeat section 4.2 for lasso with `KFold(n_splits=5, shuffle=True, random_state=s)` for `s` from 0 to 19. How much does the chosen $\lambda$ move, and how often does the one-standard-error rule keep exactly the four true features?